# SentinelPay: Baseline Model
## Notebook 03 — Logistic Regression Reference Point

**Author:** SentinelPay Research Team  
**Objective:** Establish a linear baseline to measure incremental gains from non-linear ensemble models. A baseline quantifies how much of the fraud signal is linearly separable versus requiring complex decision boundaries.

---

### 1. Why Start with a Baseline?

In applied ML research, always establish a simple baseline before deploying complex models. A Logistic Regression baseline:
- Proves that the features contain predictive signal.
- Provides a lower bound on expected performance.
- Serves as a sanity check: if XGBoost cannot beat logistic regression, the features or pipeline may be flawed.
- Is fully interpretable via learned coefficients.

In [ ]:
import pandas as pd
import numpy as np
import joblib
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, average_precision_score,
    precision_recall_curve
)
import warnings
warnings.filterwarnings('ignore')

features = ['amount', 'distance', 'time_delta', 'merchant_risk', 'device_trust',
            'velocity_1h', 'velocity_24h', 'hour_of_day', 'is_weekend']

# Load saved model and test split
scaler = joblib.load('../models/preprocessor.pkl')
baseline_model = joblib.load('../models/logistic_regression_baseline.pkl')
df_test = pd.read_csv('../data/processed/test_split.csv')

X_test_scaled = scaler.transform(df_test[features])
y_test = df_test['is_fraud']

print(f"Test Set: {len(y_test):,} records ({y_test.sum()} fraud / {(y_test == 0).sum()} legit)")

### 2. Baseline Predictions and Metrics

In [ ]:
y_prob = baseline_model.predict_proba(X_test_scaled)[:, 1]
y_pred = (y_prob >= 0.5).astype(int)

print("Classification Report (Logistic Regression Baseline):")
print("=" * 60)
print(classification_report(y_test, y_pred, target_names=['Legitimate', 'Fraud']))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

print("Confusion Matrix:")
print(f"  True Negatives:  {tn:>6,}  (Legit correctly cleared)")
print(f"  False Positives: {fp:>6,}  (Legit incorrectly flagged)")
print(f"  False Negatives: {fn:>6,}  (Fraud missed - CRITICAL)")
print(f"  True Positives:  {tp:>6,}  (Fraud correctly caught)")

print(f"\nPR-AUC:  {average_precision_score(y_test, y_prob):.4f}")
print(f"ROC-AUC: {roc_auc_score(y_test, y_prob):.4f}")

### 3. Learned Coefficients

In [ ]:
coefficients = pd.DataFrame({
    'Feature': features,
    'Coefficient': baseline_model.coef_[0],
    'Abs_Coefficient': np.abs(baseline_model.coef_[0])
}).sort_values('Abs_Coefficient', ascending=False)

print("Logistic Regression Coefficients (Feature Importance Proxy):")
print("=" * 60)
for _, row in coefficients.iterrows():
    direction = '+' if row['Coefficient'] > 0 else '-'
    bar = '#' * int(row['Abs_Coefficient'] * 5)
    print(f"  {row['Feature']:<18} {direction}{row['Abs_Coefficient']:.4f}  {bar}")

print(f"\nIntercept: {baseline_model.intercept_[0]:.4f}")

### 4. Baseline Analysis

**Findings:**
- Logistic Regression achieves strong **Recall (~95%)** but poor **Precision (~57%)**, meaning it catches most fraud but generates many false alarms.
- The high false positive rate (104 legitimate transactions incorrectly flagged) would be operationally expensive in a real banking environment.
- **PR-AUC of ~0.95** confirms the features carry strong fraud signal even in a linear model.
- The coefficient analysis shows `distance`, `device_trust`, and `time_delta` are the strongest linear predictors.

**Limitations:**
- Linear decision boundaries cannot capture interaction effects (e.g., high distance is suspicious only when combined with low device trust).
- Non-linear ensemble methods are expected to improve precision significantly.

---
*Proceed to Notebook 04: Model Training and Competition (Random Forest vs XGBoost).*